# EasyRAG Generation Failures And Guardrails

## What you will do

- trigger unsupported-answer and answer-model failure cases
- inspect abstention, fallback, and citation guardrail behavior
- compare answer settings that make the system stricter or looser

## Why this matters

A generation failure is not always a retrieval failure. Sometimes the evidence exists, but the answer layer still needs guardrails around abstention, citation discipline, and fallback behavior.


## Source code anchors

- `easyrag.rag.generation.synthesis.synthesize_answer`
- `easyrag.rag.evaluation.grounding.answer_abstained`
- `easyrag.rag.types.AnswerParam`


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.rag.evaluation.grounding import answer_abstained, answer_has_citations


## Deterministic path

We will use one answerable question and one unsupported question so the guardrail behavior has something concrete to respond to.


In [ ]:
guard_tmp = tempfile.TemporaryDirectory()
guard_rag = EasyRAG(
    working_dir=guard_tmp.name,
    workspace="guardrails-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(guard_rag.initialize_storages())
run_sync(
    guard_rag.ainsert(
        [
            "# Retrieval\nEasyRAG uses grounded retrieval and query rewrite.\n",
            "# Policy\nCitation-aware answers stay inspectable.\n",
        ],
        ids=["doc::retrieval", "doc::policy"],
        file_paths=["docs/retrieval.md", "docs/policy.md"],
    )
)
strict_param = AnswerParam(require_citations=True, allow_abstain=True, max_citations=2, max_context_chars=150)
loose_param = AnswerParam(require_citations=False, allow_abstain=False, max_citations=1, max_context_chars=80)
strict_supported = run_sync(guard_rag.aanswer("How does EasyRAG use query rewrite?", QueryParam(mode="naive", rewrite_enabled=False, mqe_enabled=False), strict_param))
strict_unsupported = run_sync(guard_rag.aanswer("When is the company retreat?", QueryParam(mode="naive", rewrite_enabled=False, mqe_enabled=False), strict_param))
loose_supported = run_sync(guard_rag.aanswer("How does EasyRAG use query rewrite?", QueryParam(mode="naive", rewrite_enabled=False, mqe_enabled=False), loose_param))
_print_json({
    "strict_supported": {"answer": strict_supported.answer, "metadata": strict_supported.metadata},
    "strict_unsupported": {"answer": strict_unsupported.answer, "metadata": strict_unsupported.metadata},
    "loose_supported": {"answer": loose_supported.answer, "metadata": loose_supported.metadata},
})


## Output inspection

The next cell simulates an answer-model failure. EasyRAG should still return a grounded fallback answer instead of crashing the whole flow.


In [ ]:
def failing_answer_model(prompt: str, **kwargs):
    raise RuntimeError("answer model unavailable")

failure_tmp = tempfile.TemporaryDirectory()
failure_rag = EasyRAG(
    working_dir=failure_tmp.name,
    workspace="guardrails-failure-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
    answer_model_func=failing_answer_model,
)
run_sync(failure_rag.initialize_storages())
run_sync(failure_rag.ainsert("# Retrieval\nEasyRAG uses grounded retrieval and query rewrite.\n", ids=["doc::retrieval"], file_paths=["docs/retrieval.md"]))
failure_answer = run_sync(failure_rag.aanswer("How does EasyRAG use query rewrite?", QueryParam(mode="naive", rewrite_enabled=False, mqe_enabled=False), strict_param))
_print_json({
    "answer": failure_answer.answer,
    "metadata": failure_answer.metadata,
    "abstained": answer_abstained(failure_answer.answer),
    "has_citations": answer_has_citations(failure_answer.answer),
})


## Provider-backed path

When providers are configured, the optional cell keeps the same guardrail inspection but runs with the repo's default retrieval wiring.


In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(working_dir=provider_tmp.name, workspace="provider-guardrails-demo")
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(provider_rag.ainsert("# Retrieval\nGrounded retrieval keeps answers inspectable.\n", ids=["doc::retrieval"], file_paths=["docs/retrieval.md"]))
        provider_answer = run_sync(provider_rag.aanswer("How do answers stay inspectable?", QueryParam(mode="hybrid", chunk_top_k=2), AnswerParam()))
        _print_json({"answer": provider_answer.answer, "metadata": provider_answer.metadata})
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()


## What changed and why

Guardrails change how the system fails, not only how it succeeds. `allow_abstain` controls whether unsupported questions should stop early. `require_citations` controls whether the final answer must stay visibly grounded. Fallback behavior controls whether answer synthesis degrades gracefully when a model call fails.


In [ ]:
run_sync(guard_rag.finalize_storages())
guard_tmp.cleanup()
run_sync(failure_rag.finalize_storages())
failure_tmp.cleanup()
print("Cleaned up the generation-guardrails workspaces.")


## Next steps

- Continue with [06_04_answer_grounding_and_faithfulness.ipynb](../06_evaluation/06_04_answer_grounding_and_faithfulness.ipynb) to score these guardrail choices systematically.
- Read [principles/20-generation-failures-and-guardrails.md](../../docs/principles/20-generation-failures-and-guardrails.md) for the conceptual framing.
- Revisit [05_04_prompting_and_answer_style.ipynb](05_04_prompting_and_answer_style.ipynb) if you want the prompt-layer side of these answer behaviors.
